In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import datetime

import numpy as np
import pandas as pd

import zipfile


In [ ]:
# unzip folder

my_zipfolder = '/content/drive/My Drive/titanic/titanic.zip'
with zipfile.ZipFile(my_zipfolder, 'r') as zip_ref:
    zip_ref.extractall('working_directory')

In [ ]:
# load raw data

filename = '/content/working_directory/train.csv'
raw_train = pd.read_csv(filename)
print('data set shape: ', raw_train.shape, '\n')
print(raw_train.head())


data set shape:  (891, 12) 

   PassengerId  Survived  Pclass  ...     Fare Cabin  Embarked
0            1         0       3  ...   7.2500   NaN         S
1            2         1       1  ...  71.2833   C85         C
2            3         1       3  ...   7.9250   NaN         S
3            4         1       1  ...  53.1000  C123         S
4            5         0       3  ...   8.0500   NaN         S

[5 rows x 12 columns]


In [ ]:
dr = ['PassengerId','Name','Ticket','Cabin','Embarked']
train = raw_train.drop(labels = dr, axis = 1)

X = train.drop('Survived', axis=1)
y = train['Survived'].values


In [ ]:
print('data set shape: ', X.shape, '\n')
print(X.head())
print(X.describe())


data set shape:  (891, 6) 

   Pclass     Sex   Age  SibSp  Parch     Fare
0       3    male  22.0      1      0   7.2500
1       1  female  38.0      1      0  71.2833
2       3  female  26.0      0      0   7.9250
3       1  female  35.0      1      0  53.1000
4       3    male  35.0      0      0   8.0500
           Pclass         Age       SibSp       Parch        Fare
count  891.000000  714.000000  891.000000  891.000000  891.000000
mean     2.308642   29.699118    0.523008    0.381594   32.204208
std      0.836071   14.526497    1.102743    0.806057   49.693429
min      1.000000    0.420000    0.000000    0.000000    0.000000
25%      2.000000   20.125000    0.000000    0.000000    7.910400
50%      3.000000   28.000000    0.000000    0.000000   14.454200
75%      3.000000   38.000000    1.000000    0.000000   31.000000
max      3.000000   80.000000    8.000000    6.000000  512.329200


In [ ]:
# count missing values
X.isna().sum()

Pclass      0
Sex         0
Age       177
SibSp       0
Parch       0
Fare        0
dtype: int64

In [ ]:
import numpy as np
import warnings
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion, Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')


# Custom Transformer that fills missing ages
class CustomImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        super().__init__()
        self.age_means_ = {}

    def fit(self, X, y=None):
        self.age_means_ = X.groupby(['Pclass', 'Sex']).Age.mean()

        return self

    def transform(self, X, y=None):
        # fill Age
        for key, value in self.age_means_.items():
            X.loc[((np.isnan(X["Age"])) & (X.Pclass == key[0]) & (X.Sex == key[1])), 'Age'] = value

        return X


class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        super().__init__()
        self.means_ = None
        self.std_ = None

    def fit(self, X, y=None):
        X = X.to_numpy()
        self.means_ = X.mean(axis=0, keepdims=True)
        self.std_ = X.std(axis=0, keepdims=True)

        return self

    def transform(self, X, y=None):
        X[:] = (X.to_numpy() - self.means_) / self.std_

        return X


class CategoricalTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        super().__init__()

    # Return self nothing else to do here
    def fit(self, X, y=None):
        return self

    # Helper function that converts values to Binary depending on input
    def create_binary(self, obj):
        if obj == 0:
            return 'No'
        else:
            return 'Yes'

    # Transformer method for this transformer
    def transform(self, X, y=None):
        # Categorical features to pass down the categorical pipeline
        return X[['Sex']].values


class NumericalTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        super().__init__()

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        # Numerical features to pass down the numerical pipeline
        X = X[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']]
        X = X.replace([np.inf, -np.inf], np.nan)
        return X.values


# Defining the steps in the categorical pipeline
categorical_pipeline = Pipeline(steps=[
    ('cat_transformer', CategoricalTransformer()),
    ('one_hot_encoder', OneHotEncoder(sparse=False))])

# Defining the steps in the numerical pipeline
numerical_pipeline = Pipeline(steps=[
    ('num_transformer', NumericalTransformer()),
    ('imputer', SimpleImputer(strategy='median')),
    ('std_scaler', StandardScaler())])

# Combining numerical and categorical pipeline into one full big pipeline horizontally
# using FeatureUnion
union_pipeline = FeatureUnion(transformer_list=[
    ('categorical_pipeline', categorical_pipeline),
    ('numerical_pipeline', numerical_pipeline)])

# Combining the custom imputer with the categorical and numerical pipeline
preprocess_pipeline = Pipeline(steps=[('custom_imputer', CustomImputer()),
                                      ('full_pipeline', union_pipeline)])




In [ ]:
# TRY THE CUSTOM IMPUTTER
# Resetting dataset
train = raw_train.drop(labels = dr, axis = 1)
X = train.drop('Survived', axis=1)

print('--> Before applying transform')
print('Nan count:', X.isna().sum())
print('Age mean:', X['Age'].mean(), '\n')

# Instantiating Instantiating
my_CustomImputer = CustomImputer()

# Fitting operation
my_CustomImputer.fit(X)

print('Learned parameters')
print(my_CustomImputer.age_means_)

# Applying the CustomImputer
my_CustomImputer.transform(X)

print('\n--> After applying transform')
print('Nan count:', X.isna().sum())
print('Age mean:', X['Age'].mean())

--> Before applying transform

Nan count: Pclass      0
Sex         0
Age       177
SibSp       0
Parch       0
Fare        0
dtype: int64
Age mean: 29.69911764705882 

Learned parameters
Pclass  Sex   
1       female    34.611765
        male      41.281386
2       female    28.722973
        male      30.740707
3       female    21.750000
        male      26.507589
Name: Age, dtype: float64

--> After applying transform
Nan count: Pclass    0
Sex       0
Age       0
SibSp     0
Parch     0
Fare      0
dtype: int64
Age mean: 29.31864271664414


In [ ]:
# TRY THE CUSTOM SCALER
print('--> Before applying transform')
print(X['Age'].describe(), '\n')

# Instantiating Instantiating
my_CustomScaler = CustomScaler()

# Fitting operation
my_CustomScaler.fit(X['Age'])

print('Learned parameters')
print('mean:', my_CustomScaler.means_ , '\n', 'std:', my_CustomScaler.std_)

print(my_CustomScaler.get_params(deep=True))

# Applying the CustomImputer
my_CustomScaler.transform(X['Age'])

print('\n--> After applying transform')
print(X['Age'].describe())

--> Before applying transform
count    8.910000e+02
mean    -2.164997e-17
std      1.000562e+00
min     -2.177144e+00
25%     -5.702007e-01
50%     -2.117770e-01
75%      5.033550e-01
max      3.818194e+00
Name: Age, dtype: float64 

Learned parameters
mean: [0.] 
 std: [1.]
{}

--> After applying transform
count    8.910000e+02
mean    -2.164997e-17
std      1.000562e+00
min     -2.177144e+00
25%     -5.702007e-01
50%     -2.117770e-01
75%      5.033550e-01
max      3.818194e+00
Name: Age, dtype: float64


In [ ]:
# MODEL

from sklearn import tree

# Decision Tree
decision_tree = tree.DecisionTreeClassifier()

In [ ]:
# define full pipeline --> preprocessing + model
full_pipeline = Pipeline(steps=[
    ('preprocess_pipeline', preprocess_pipeline),
    ('model', decision_tree)])

# fit on the complete pipeline
training = full_pipeline.fit(X, y)
print(full_pipeline.get_params())

# metrics
score_test = \
    round(training.score(X, y) * 100, 2)
print(f"\nTraining Accuracy: {score_test}")

{'memory': None, 'steps': [('preprocess_pipeline', Pipeline(memory=None,
         steps=[('custom_imputer', CustomImputer()),
                ('full_pipeline',
                 FeatureUnion(n_jobs=None,
                              transformer_list=[('categorical_pipeline',
                                                 Pipeline(memory=None,
                                                          steps=[('cat_transformer',
                                                                  CategoricalTransformer()),
                                                                 ('one_hot_encoder',
                                                                  OneHotEncoder(categories='auto',
                                                                                drop=None,
                                                                                dtype=<class 'numpy.float64'>,
                                                                                handle_un

In [ ]:
# Prediction

my_data = X.iloc[[77]]
y = full_pipeline.predict(my_data)
print(my_data, y)





    Pclass   Sex       Age  SibSp  Parch  Fare
77       3  male -0.211777      0      0  8.05 [0]
